# Unsloth Fine-Tuning on CUDA

Fine-tune a Llama 3.2 model using Unsloth + LoRA on a CUDA GPU.

- Model loaded directly from Hugging Face via Unsloth's `FastLanguageModel`
- Dataset fetched directly from Hugging Face Hub
- All metrics logged to **TensorBoard**
- Requires CUDA GPU (Unsloth does not support CPU/MPS)


In [ ]:
# Install required packages (CUDA environment)
%pip install -q unsloth datasets trl tensorboard huggingface_hub seaborn


# Step 1: Load Model

Load `unsloth/Llama-3.2-3B` directly from Hugging Face using Unsloth's `FastLanguageModel`.
Unsloth automatically patches the model for 2x faster training with reduced VRAM usage.


In [ ]:
from unsloth import FastLanguageModel

HF_MODEL_ID = "unsloth/Llama-3.2-3B"
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=HF_MODEL_ID,
    max_seq_length=max_seq_length,
    dtype=None,          # auto-detect bfloat16 / float16
    load_in_4bit=True,   # 4-bit quantisation for lower VRAM
)

print(f"✓ Loaded: {HF_MODEL_ID}")
print(f"✓ Max sequence length: {max_seq_length}")


# Step 2: Attach LoRA Adapters

Attach LoRA adapters using Unsloth's optimised `get_peft_model`. Only the adapter weights are trained, keeping VRAM usage low.

Parameters: rank=16, alpha=16.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # saves extra VRAM
    random_state=3407,
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✓ LoRA adapters attached")
print(f"✓ Trainable parameters: {trainable_params:,}")


# Step 3: Load Dataset

Load the `ServiceNow-AI/R1-Distill-SFT` dataset directly from Hugging Face Hub.
A subset of 5120 examples is used by default; adjust `subset_size` as needed.


In [ ]:
from datasets import load_dataset

HF_DATASET_ID = "ServiceNow-AI/R1-Distill-SFT"

dataset = load_dataset(HF_DATASET_ID, "v0", split="train")

# Use a subset for faster iteration; set to None to use the full dataset
subset_size = 5120
if subset_size and subset_size < len(dataset):
    dataset = dataset.select(range(subset_size))

print(f"✓ Dataset loaded: {len(dataset):,} examples")
print(f"✓ Columns: {dataset.column_names}")
print(f"\nSample problem snippet:\n{dataset[0].get('problem', '')[:300]}...")


# Step 4: Format Dataset

Apply the Llama 3.1 chat template to each example.
Each row is converted into a user/assistant conversation and tokenised to match inference-time format.


In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

def formatting_prompts_func(examples):
    texts = []
    for problem, response in zip(examples["problem"], examples["reannotated_assistant_content"]):
        convo = [
            {"role": "user", "content": problem},
            {"role": "assistant", "content": response},
        ]
        texts.append(tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False))
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

print("✓ Dataset formatted with chat template")
print(f"\nFormatted example preview:\n{dataset[0]['text'][:300]}...")


# Step 4a: Exploratory Data Analysis

Quick inspection of the formatted dataset before training:

1. **Sample comparison** — raw vs formatted text for the first example
2. **Context length distribution** — token count of the full formatted prompt
3. **Response length distribution** — token count of the assistant reply only


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

sns.set_theme(style="whitegrid", context="talk", palette="muted")

# ── 1. Sample: raw vs formatted ──────────────────────────────────────────────
sample = dataset[0]
raw_problem = sample["problem"]
raw_response = sample["reannotated_assistant_content"]
formatted_text = sample["text"]

print("=" * 70)
print("RAW PROBLEM (first 300 chars):")
print(raw_problem[:300])
print("\nRAW RESPONSE (first 300 chars):")
print(raw_response[:300])
print("\n" + "=" * 70)
print("FORMATTED TEXT (first 600 chars):")
print(formatted_text[:600])
print("=" * 70)

# ── 2 & 3. Compute lengths ───────────────────────────────────────────────────
def token_len(text: str) -> int:
    return len(tokenizer.encode(text, add_special_tokens=False))

print("\nComputing token lengths (this may take ~1 min for 10k examples)...")

context_lengths = [token_len(row["text"]) for row in dataset]
response_lengths = [token_len(row["reannotated_assistant_content"]) for row in dataset]

df = pd.DataFrame({
    "context_length": context_lengths,
    "response_length": response_lengths,
})

print(f"✓ Done. Computed lengths for {len(df):,} examples.\n")
print(df[["context_length", "response_length"]].describe().round(1).to_string())

# ── 2. Context Length Distribution ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

sns.histplot(
    data=df,
    x="context_length",
    bins=60,
    kde=True,
    color="steelblue",
    ax=axes[0],
)
axes[0].set_title("Distribution of Context Lengths")
axes[0].set_xlabel("Context Length (tokens)")
axes[0].set_ylabel("Frequency")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

# ── 3. Response Length Distribution ──────────────────────────────────────────
sns.histplot(
    data=df,
    x="response_length",
    bins=60,
    kde=True,
    color="coral",
    ax=axes[1],
)
axes[1].set_title("Distribution of Response Lengths")
axes[1].set_xlabel("Response Length (tokens)")
axes[1].set_ylabel("Frequency")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

plt.suptitle("Token Length Distributions", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()


# Step 5: Configure Trainer

Configure TRL's `SFTTrainer`. Metrics are reported to **TensorBoard** (`./outputs/logs`).
Launch TensorBoard with: `tensorboard --logdir outputs/logs`


In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments, DataCollatorForLanguageModeling
import torch

MAX_STEPS_QUICK_RUN = -1          # set to a positive number for a quick smoke-test
PER_DEVICE_TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4  # effective batch size = 8
WARMUP_STEPS = 5
NUM_TRAIN_EPOCHS = 3
LEARNING_RATE = 2e-4

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    data_collator = DataCollatorForLanguageModeling(tokenizer = tokenizer, mlm=False),
    dataset_num_proc=2,
    packing = False,  # Can make training 5x faster for short sequences
    args=TrainingArguments(
        output_dir="outputs",
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        warmup_steps=WARMUP_STEPS,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        max_steps=MAX_STEPS_QUICK_RUN,
        learning_rate=LEARNING_RATE,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        report_to="tensorboard",
        logging_dir="outputs/logs",
    ),
)

print("✓ Trainer configured")
print(f"✓ Reporting metrics to TensorBoard → ./outputs/logs")


# Step 6: Train the Model

VRAM stats are printed before training starts. All metrics (loss, learning rate, throughput) are written to TensorBoard in real time.


In [ ]:
import torch

gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_memory = round(gpu_stats.total_memory / 1024**3, 3)
print(f"Device: {gpu_stats.name}  |  Total VRAM: {max_memory} GB  |  Reserved before training: {start_gpu_memory} GB\n")

trainer_stats = trainer.train()

used_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
print(f"\n✓ Training complete!")
print(f"Final loss:       {trainer_stats.training_loss:.4f}")
print(f"Peak VRAM:        {used_memory} GB ({round(used_memory / max_memory * 100, 1)}% of {max_memory} GB)")
print(f"Training delta:   {round(used_memory - start_gpu_memory, 3)} GB")
print(f"\nView metrics:  tensorboard --logdir runs")


# Step 7: Inference Sanity Check

Switch the model to inference mode and run a quick reasoning prompt to verify the fine-tuned weights.


In [ ]:
from unsloth import FastLanguageModel
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

test_problem = (
    "If Alex is taller than Blake, Blake is taller than Casey, "
    "and Casey is taller than Dana, who is the shortest person?"
)

messages = [{"role": "user", "content": test_problem}]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(
    input_ids=inputs,
    streamer=text_streamer,
    max_new_tokens=1024,
    use_cache=True,
    temperature=0.7,
    top_p=0.9,
)


# Step 8: Save Artifacts

Save the LoRA adapter weights locally. Optionally push to the Hugging Face Hub or export to GGUF.


In [ ]:
model.save_pretrained("outputs/adapters/")
tokenizer.save_pretrained("outputs/adapters/")
print("✓ LoRA adapters saved to ./outputs/adapters/")

from huggingface_hub import notebook_login
notebook_login()

# Optional: push to Hugging Face Hub
model.push_to_hub("tangowhisky16/llama-3.2-3B-think")
tokenizer.push_to_hub("tangowhisky16/llama-3.2-3B-think")

api.upload_folder(
    folder_path="outputs",
    repo_id="tangowhisky16/llama-3.2-3B-think",
    repo_type="model"
)

# Optional: export to GGUF (requires llama.cpp)
# model.save_pretrained_gguf("lora_model_gguf", tokenizer, quantization_method="q4_k_m")
